# Handwritten Document Pipeline — Prep + Transcription
---
**This notebook is the Google Colab version.** The same pipeline also exists (or will exist) as
plain-Jupyter, GitHub, and local-Ollama-desktop variants — those mirror the logic here, just
swapping out the Colab-specific bits (Drive mount, `!pip`/`!sudo apt-get` cells, the file-upload
widget). If you're editing pipeline *logic*, consider whether the change should carry over to
those versions too.

One notebook, up to three stages:

1. **Image Prep** — take a zip, a PDF, standalone images, or a folder of images; unzip/convert as
   needed, fix orientation, resize to a consistent size, optionally boost contrast/sharpness, save
   to your chosen storage
2. **Transcription** — run each prepped page through a vision model in Ollama, with automatic
   retry on empty results and the ability to fall back to a different model
3. **Cataloguing (optional)** — draft an item-level ISAD(G) catalogue record for each page from
   its transcription: title, dates, extent, creator, scope and content, language/scripts, physical
   characteristics, access points, a confidence estimate per field, and points flagged for human
   review (including potential sensitivity). Toggle on/off with `GENERATE_SUMMARIES` in the config
   cell.

Runtime: **T4 GPU** (Runtime > Change runtime type).

### How to use this notebook
Everything you might want to change lives in **one configuration cell** right below this. Set
that up, then `Runtime > Run all`. You shouldn't need to touch anything past that cell unless
you're troubleshooting or changing pipeline logic.

### What changed in v5

This version targets a specific failure mode: **dense, full pages of continuous handwriting
coming back as `[illegible]` or empty**, while short/sparse text on the same kind of page
transcribes fine. Diagnosis: a vision model's per-image "reading budget" gets spent much faster
on a dense page than a sparse one, and — especially at `TEMPERATURE = 0.0` — a model that fails
once will fail identically on every retry, since there's nothing to make it try something
different. Three changes address this:

1. **Retries now escalate temperature** (`RETRY_TEMPERATURE`) after the first attempt, so a
   retry can actually produce a different result instead of repeating the same failure.
2. **Auto-tiling for pages that still look weak** — if a page's result is short/empty, or its
   *visual ink coverage* suggests more writing than the transcription reflects, the page is
   automatically split into overlapping horizontal bands and each band is transcribed
   separately, then stitched back together. This is a direct fix for the "wall of dense text"
   problem, applied only when needed so easy pages stay fast.
3. **Ink-coverage-aware QA** — Stage 2.5 now flags pages where the image visibly has a lot of
   dark pixels but the transcription (once `[illegible]` placeholders are stripped) is thin,
   not just pages under `MIN_CHARS`. This catches "looks done but isn't" cases the old flat
   character-count check missed.

See the new **TRANSCRIPTION RETRY & TILING** block in the config cell for the tunable settings,
and `transcription_log.csv` (written alongside the `.txt` files) for a per-page record of which
method/model actually produced each result.


### Patch on top of v5: repetition-loop guard

Testing (minicpm-v) surfaced a second, distinct failure mode -- most visible on sparse/near-blank
pages: the model runs out of genuine content and gets stuck at `TEMPERATURE = 0.0` repeating the
same short word or placeholder hundreds of times (`"di. di. di. ..."`, `"[illegible] [illegible]
..."`) until it hits the token limit. This is worse than an honest short/empty result because it's
long enough to clear `MIN_CHARS` and the dense-page ink check above, so it passed QA silently.

Fixed at the shared layer (`call_model`), so both full-page and tiled calls inherit it automatically:
- `REPEAT_PENALTY` / `REPEAT_LAST_N` added to the Ollama call options, to discourage looping at
  the sampling level in the first place.
- `truncate_at_repetition()` detects a short word-chunk repeating `LOOP_MIN_REPEATS`+ times and
  trims the output back to whatever genuine content came before the loop, rather than discarding
  the whole page.
- `SPARSE_NUM_PREDICT_CAP` caps the prediction budget on pages with low ink coverage --
  specifically where this failure has been observed -- as a second line of defence.
- Stage 2.5's QA now also flags pages where a loop was detected/trimmed, read from
  `transcription_log.csv`'s new `loop_detected` column.


## Configuration
---
Everything you can change is here, in one cell, grouped by section. Set this up once, then
`Runtime > Run all`.

- **Storage** — where prepped images and transcriptions get saved
- **Input source** — where your source material (zip / PDF / images) is coming from
- **Prep settings** — resizing, contrast, sharpening for Stage 1
- **Transcription settings** — which model(s), sampling, retry behaviour for Stage 2
- **Cataloguing settings** — on/off switch, model, and record format for Stage 3 (optional)

Everything below this cell reads from these variables — you shouldn't need to edit anything
past this point for a normal run.


In [1]:
# =============================================================================
# STORAGE — where prepped images and transcriptions are saved
# =============================================================================
STORAGE_BACKEND = "drive"   # "drive" or "onedrive"

ONEDRIVE_CLIENT_ID = ""     # <-- only needed if STORAGE_BACKEND = "onedrive"
                             #     (Azure app registration — see the "Connect" cell below
                             #     for one-time setup instructions)

# =============================================================================
# INPUT SOURCE — where your source material is coming from
# =============================================================================
# Set UPLOAD_INSTEAD = True to skip the path settings below entirely and instead get a
# file-picker widget when you run the "Get your images" cell. You can select a zip, a PDF,
# one image, or several images at once — the notebook figures out what you gave it.
UPLOAD_INSTEAD = False

# If UPLOAD_INSTEAD is False, point SOURCE_PATH / ONEDRIVE_SOURCE_PATH at your source.
# All of these are supported, auto-detected from the path/extension:
#   - a .zip file containing images
#   - a .pdf file (each page becomes one image)
#   - a folder containing loose images (.jpg/.jpeg/.png/.webp)
#   - a single loose image file

SOURCE_PATH = "/content/drive/MyDrive/S-260706-142934.zip"
# ^ used when STORAGE_BACKEND = "drive" — path to your zip/pdf/image/folder in Drive

ONEDRIVE_SOURCE_PATH = "/PipelineInput/S-260706-142934.zip"
# ^ used when STORAGE_BACKEND = "onedrive" — path to your zip/pdf/image/folder in OneDrive

PDF_DPI = 300   # only used if your source is a PDF — resolution pages are rasterised at
                # before Stage 1 resizes them down to target_long_edge. 300 is a good default;
                # raise it if source pages are very large/dense, lower it to save time.

# =============================================================================
# PREP SETTINGS (Stage 1)
# =============================================================================
# - target_long_edge resizes every page so its LONGEST side hits this many pixels, regardless
#   of the original size. This keeps every page's token footprint consistent going into the
#   model — a fixed percentage doesn't, since source pages can vary in size. 1600px is a
#   reasonable starting point; raise it if fine handwriting is still hard to read, lower it if
#   you hit context-size errors during transcription.
# - enhance_contrast / contrast_mode — "auto" uses PIL's autocontrast, which normalizes each
#   image against its own pixel histogram (good for faint pencil, adapts per-page). "fixed"
#   applies the same multiplier to every page — simpler, but can under- or over-correct
#   depending on the page.
# - sharpen is optional — can help fine pen strokes survive resizing, but test it, don't
#   assume it helps.

max_size_bytes = 10 * 1024 * 1024  # 10 MB cap per output file

target_long_edge = 1600      # pixels — longest side of the image after resize
                              # set to a very high number (e.g. 99999) to effectively disable resizing

enhance_contrast = True      # set False to skip contrast enhancement entirely
contrast_mode = "fixed"       # "auto" (adaptive per-image) or "fixed" (same factor every time)
contrast_factor = 1.4        # only used when contrast_mode = "fixed"
autocontrast_cutoff = 1      # only used when contrast_mode = "auto"; trims extreme 1% of pixels before stretching

sharpen = False               # optional light sharpening after resize (set to false - if unnecessary)

save_name = "Interesting_Doc"  # <-- change this to whatever name you want
                                              #     prepped images are saved under

# =============================================================================
# TRANSCRIPTION SETTINGS (Stage 2)
# =============================================================================
# MODEL_CANDIDATES is tried IN ORDER for each page: if the first model fails or comes back
# empty/short after all its retries, the next one in the list is tried. This is also how you
# switch models entirely (just leave one model in the list) or compare models on the same
# batch (leave several in — the transcription log records which model produced each result).
#
# Every model listed here gets pulled automatically in the "Pull models" cell below. Each is
# a multi-GB download and uses Colab's disk + GPU memory, so only list what you actually plan
# to use.
#
# Options worth knowing about:
#   "qwen3-vl"                     — solid general-purpose vision model, good default
#   "minicpm-v"                    — strong OCR benchmark results, especially dense/handwritten text
#   "llama3.2-vision"              — solid general-purpose OCR, widely used (currently doesn't seem to be working)
#   "maternion/lightonocr-2"       — lightweight (1B) — fast, good quick first-pass/fallback
#   "gemma3"                       — newer general-purpose multimodal model
#   "deepseek-ocr"
#   "glm-ocr"
# for some of these runtimes are exceedingly long - simply go to the drop arrow next to "run all" and click "restart session"
# this will clear everything and you change your model candidates and press "run all" again

#
# v5: defaults to TWO models rather than one. With a single model, "fall back to the next
# model" never actually happens — a page that genuinely defeats that model just retries itself
# into the ground. minicpm-v is listed second because it benchmarks well on dense/handwritten
# text specifically, which is exactly the failure mode this version targets.
# sub out any of the models which are on ollama - dependent upon their size

MODEL_CANDIDATES = ["glm-ocr","minicpm-v"] #put the model you are testing as first in this list

TEMPERATURE = 0.0      # first-attempt temperature. Kept at 0.0 for reproducibility on easy pages.
NUM_CTX = 8192        # Ollama's default (4096) is too small for a full-res image + prompt +
                       # response, and causes errors or silent empty (0 chars) results
NUM_PREDICT = 2048

MIN_CHARS = 20         # transcriptions shorter than this are treated as failed/empty
MAX_ATTEMPTS = 5       # retries per model before falling back to the next one in MODEL_CANDIDATES

BATCH_SIZE = 50        # pages processed per run of the transcription cell — re-run the cell
                        # to continue with the next batch

PROMPT = (
    "Transcribe the handwritten text in this image exactly as written. "
    "Preserve original spelling, punctuation, and line breaks where possible. "
    "Do not paraphrase, summarize, correct, or add commentary of any kind. "
    "If a word or short phrase is illegible, write [illegible]. "
    "If a whole line is illegible, write [illegible line]. "
    "Once you have transcribed everything visible, stop — do not repeat any word, phrase, "
    "or placeholder. If the page or section is blank, write [blank] once and stop. "
    "Output ONLY the transcription itself — no preamble, no notes, no explanation."
)

# change this prompt if need be - possibly requires research into prompt engineering (but i would rather gauge my eyes out)

# =============================================================================
# TRANSCRIPTION RETRY & TILING (Stage 2, v5)
# =============================================================================
# Why this section exists: a vision model's ability to resolve individual characters degrades
# as a page gets denser (more lines, less space per line) even when the pixel resolution and
# contrast are identical to an easy page. Small/sparse pages transcribe fine; a full page of
# continuous handwriting is a much harder task for the same model, and it can fail entirely
# (empty output, or a blanket "[illegible]") rather than partially. These settings give the
# pipeline a way to notice that and retry smarter, rather than retrying the same way and
# getting the same result every time.

RETRY_TEMPERATURE = 0.2


ENABLE_AUTO_TILING = True
# If a page's transcription still looks weak after exhausting MODEL_CANDIDATES (see
# needs_tiling() in Stage 2.4), the page is automatically split into overlapping horizontal
# bands, each band is transcribed independently, and the results are stitched back together.
# This directly targets dense pages: each tile has far fewer lines to resolve than the full
# page did, at the same pixel density. Set False to disable and keep the old whole-page-only
# behaviour.

TILE_COUNT = 3            # number of horizontal bands to split a weak/dense page into
TILE_OVERLAP_FRAC = 0.08  # overlap between adjacent tiles, as a fraction of tile height —
                           # prevents a line that straddles a tile boundary from being lost
                           # entirely (some duplication at seams is possible; spot-check tiled
                           # pages rather than assuming a clean stitch)

# --- Ink-coverage-aware dense-page detection ---
# A flat MIN_CHARS check misses a real failure mode: a page that visibly has plenty of writing
# but comes back mostly/entirely "[illegible]" — that can clear MIN_CHARS on character count
# alone while containing almost no real transcribed content. Ink coverage (% of dark pixels in
# the image) is a cheap, page-content-agnostic way to estimate "how much writing is actually on
# this page" independent of what the model returned, so it can catch that mismatch.
INK_DARK_PIXEL_THRESHOLD = 150   # grayscale value (0-255) below which a pixel counts as "ink"
DENSE_INK_THRESHOLD_PCT = 1.8    # ink coverage % above which a page is considered visually dense
                                  # calibrated from two real samples: a short/sparse page that
                                  # transcribed fine measured ~1.0%, a full dense page that
                                  # failed measured ~2.6% at INK_DARK_PIXEL_THRESHOLD=150 —
                                  # 1.8 sits between them. Recalibrate for your own collection if
                                  # pages are unusually light/dark or have heavy foxing/staining.
DENSE_PAGE_MIN_CHARS = 100       # if a page is flagged dense but its real content (after
                                  # stripping [illegible] placeholders) is under this many
                                  # characters, it's treated as needing tiling/review even
                                  # though it cleared the plain MIN_CHARS check.

# =============================================================================
# REPETITION-LOOP GUARDS (Stage 2)
# =============================================================================
# A different failure mode from the dense-page one above: on SPARSE or near-blank pages, a
# small vision model can run out of genuine content to describe and then get stuck at
# TEMPERATURE = 0.0 (deterministic, no exploration) reproducing the same short word/placeholder
# over and over until it hits NUM_PREDICT — e.g. "N. di. di. di. di. ..." or
# "[illegible] [illegible] [illegible] ..." repeated hundreds of times. This is worse than an
# honest short/empty result: it's long enough to clear MIN_CHARS and the dense-page ink check
# above, so it can look like a completed, confident transcription and pass QA silently. These
# settings guard against it at the source (sampling options), and as a safety net (detecting
# and trimming a loop after the fact, keeping whatever genuine content came before it).

REPEAT_PENALTY = 1.3     # Ollama's repeat_penalty (default 1.1) -- discourages the model from
                          # reproducing recent tokens. Raised here specifically to reduce the
                          # loops described above; tune down if it starts distorting legitimate
                          # repeated words/phrases in real handwriting.
REPEAT_LAST_N = 256      # how many recent tokens the repeat penalty looks back over

LOOP_MIN_REPEATS = 8       # a chunk of words repeating this many times consecutively is
                            # treated as a runaway loop rather than legitimate repetition
LOOP_CHUNK_MAX_WORDS = 6   # longest word-chunk length checked for repetition (both observed
                            # loops were 1-word chunks -- "di." / "[illegible]" -- but longer
                            # repeating phrases are checked too, up to this length)

SPARSE_INK_THRESHOLD_PCT = 1.2   # ink coverage % at/below which a page is treated as sparse —
                                   # calibrated near the ~1.0% measured on a real sparse page
                                   # that transcribed fine. Sparse pages are exactly where the
                                   # repetition loop has been observed, so they get a much
                                   # smaller prediction budget as a second line of defence.
SPARSE_NUM_PREDICT_CAP = 300      # NUM_PREDICT is capped to this on pages at/below
                                   # SPARSE_INK_THRESHOLD_PCT, so even an undetected loop can't
                                   # run far before hitting the token limit.

# =============================================================================
# CATALOGUING SETTINGS (Stage 3 — optional)
# =============================================================================
# When enabled, Stage 3 reads each page's transcription and drafts an item-level ISAD(G)
# catalogue record from it: title, dates, extent, creator, scope and content, language/scripts,
# physical characteristics, access points (persons/places/subjects/organisations), a confidence
# estimate per field, and a list of points flagged for human review.
#
# This step reads ONLY the transcribed text, not the original image — so anything that needs
# visual inspection (physical characteristics, script style, true dimensions) should come back
# blank with a "Low" confidence + a verification flag rather than a guess. Every record is a
# DRAFT — a human should review it, especially anything flagged.

GENERATE_SUMMARIES = False   # <-- set True to enable Stage 3

# Pages with no real transcribed content (empty, or just "[illegible]"/"[illegible line]"
# placeholders) are SKIPPED entirely rather than catalogued — sending a blank/illegible page
# to the summary model would just produce a fabricated-looking ISAD(G) record. This is separate
# from Stage 2's MIN_CHARS/MAX_ATTEMPTS: it's fine (and expected) for the transcription step to
# land on an empty or "[illegible]" result after exhausting its retries — that's Stage 2 doing
# its job on a genuinely blank/illegible page. This setting just stops that result from being
# catalogued. After stripping out any "[illegible ...]" placeholders, a page needs at least this
# many characters of actual transcribed text left to be catalogued.
CATALOGUE_MIN_CONTENT_CHARS = 15

# A separate, lighter, TEXT-ONLY model — no need for vision capability here, and it's a second
# model call per page on top of transcription, so a smaller/faster model keeps total runtime
# down. qwen2.5:7b is a solid default for structured extraction; swap for anything else you can
# pull in Ollama (e.g. "llama3.1:8b", "gemma2:9b", "phi3" for something smaller/faster).
SUMMARY_MODEL = "qwen2.5:7b"

SUMMARY_TEMPERATURE = 0.0
SUMMARY_NUM_CTX = 4096        # raise if you hit context errors on very long transcriptions
SUMMARY_NUM_PREDICT = 1024    # response length allowance -- the JSON record itself
SUMMARY_MAX_ATTEMPTS = 3      # retries if the model doesn't return valid, parseable JSON

# Instructions sent to the model along with each page's transcribed text. Edit the field
# descriptions here if you want different fields -- just keep the final JSON shape (or update
# the code in Stage 3 that reads these keys to match).
SUMMARY_INSTRUCTIONS = """You are an archivist producing a DRAFT item-level ISAD(G) catalogue record from a single transcribed handwritten page. You are only given the transcribed text below, not the original image, so leave fields blank rather than guessing anything you cannot support from the text, and flag anything a human cataloguer needs to check by looking at the actual image.

Fields:
- title: a short, descriptive title for this item (a label, not a summary -- e.g. "Letter to Maria, describing crossing to Genoa")
- dates: dates as they appear in the text ("as_written") and, if inferable, a normalised form ("normalised", e.g. "1943-01-03", or an approximate range like "1943" if only a year is given)
- extent: physical extent, e.g. "1 page" -- you may state page count but not physical dimensions, since you cannot see the original
- creator: the presumed author/writer of this item, only if identifiable from the text itself (e.g. a signature, a return address, an internal reference) -- otherwise leave blank
- scope_and_content: a concise (2-4 sentence) summary of what the page actually contains
- language_scripts: language(s) the text is written in; leave script/handwriting-style notes blank, since that requires seeing the image
- physical_characteristics: leave blank unless the text itself explicitly describes the physical item (e.g. "written on the back of a photograph") -- this normally requires the image, which you don't have
- access_points: names of persons, places, subjects/topics, and organisations mentioned, as separate lists
- confidence: for each of the fields above, one of "High", "Medium", or "Low", reflecting how confident you are in what you filled in (or that leaving it blank is correct)
- flags: points a human cataloguer should check before this record is used. Use "type": "verification" for anything uncertain (ambiguous names/dates, illegible sections affecting a field, unclear authorship). Use "type": "sensitivity" for anything that might be personally sensitive -- health information, legal matters, information about living or recently-deceased individuals, distressing content, or anything else that might warrant an access restriction.

Respond with ONLY a single valid JSON object, no markdown code fences, no commentary before or after, matching exactly this shape (use "" for unknown strings, [] for unknown lists -- never omit a key):

{
  "title": "",
  "dates": {"as_written": "", "normalised": ""},
  "extent": "",
  "creator": "",
  "scope_and_content": "",
  "language_scripts": "",
  "physical_characteristics": "",
  "access_points": {"persons": [], "places": [], "subjects": [], "organisations": []},
  "confidence": {"title": "", "dates": "", "extent": "", "creator": "", "scope_and_content": "", "language_scripts": "", "physical_characteristics": "", "access_points": ""},
  "flags": [{"type": "verification", "note": ""}]
}"""


## Setup — connect to storage
---
Uses the `STORAGE_BACKEND` you set above.

**Google Drive** connects with one click (a permissions popup) — no setup required.

**OneDrive** needs a one-time setup because Colab has no native OneDrive mount, so this
notebook talks to it through Microsoft's Graph API instead:
1. Go to [portal.azure.com](https://portal.azure.com) > **App registrations** > **New registration**
2. Name it anything (e.g. "Colab Pipeline"), leave redirect URI blank, register
3. On the app's **Authentication** page, set **"Allow public client flows"** to **Yes**
4. On **API permissions**, add Microsoft Graph > Delegated > `Files.ReadWrite.All`, then `Add permissions`
5. Copy the **Application (client) ID** from the app's Overview page into `ONEDRIVE_CLIENT_ID` in the config cell above

You only do this once — the same client ID works for every future session.

For OneDrive, running this cell prints a short code and a link — visit the link on any device,
enter the code, sign in. This cell also defines the upload/download helpers used later in the
notebook.


In [ ]:
import os

onedrive_access_token = None
GRAPH_BASE = "https://graph.microsoft.com/v1.0/me/drive/root:"

if STORAGE_BACKEND == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    print("Connected to Google Drive.")

elif STORAGE_BACKEND == "onedrive":
    !pip install -q msal requests
    import msal, requests

    if not ONEDRIVE_CLIENT_ID:
        raise ValueError("Set ONEDRIVE_CLIENT_ID in the config cell above first — see setup instructions.")

    GRAPH_SCOPES = ["Files.ReadWrite.All", "offline_access"]
    app = msal.PublicClientApplication(
        ONEDRIVE_CLIENT_ID,
        authority="https://login.microsoftonline.com/consumers"
    )

    flow = app.initiate_device_flow(scopes=GRAPH_SCOPES)
    if "user_code" not in flow:
        raise Exception(f"Device flow failed to start: {flow}")
    print(flow["message"])  # "To sign in, visit https://microsoft.com/devicelogin and enter code XXXXX"

    token_result = app.acquire_token_by_device_flow(flow)  # blocks until you complete sign-in

    if "access_token" not in token_result:
        raise Exception(f"Auth failed: {token_result.get('error_description')}")

    onedrive_access_token = token_result["access_token"]
    print("Connected to OneDrive.")

else:
    raise ValueError('STORAGE_BACKEND must be "drive" or "onedrive"')


def _graph_headers():
    return {"Authorization": f"Bearer {onedrive_access_token}"}

def onedrive_download_file(remote_path, local_path):
    import requests
    url = f"{GRAPH_BASE}{remote_path}:/content"
    r = requests.get(url, headers=_graph_headers())
    r.raise_for_status()
    os.makedirs(os.path.dirname(local_path) or ".", exist_ok=True)
    with open(local_path, "wb") as f:
        f.write(r.content)

def onedrive_upload_folder(local_dir, remote_folder):
    import requests
    for fname in sorted(os.listdir(local_dir)):
        local_path = os.path.join(local_dir, fname)
        if not os.path.isfile(local_path):
            continue
        url = f"{GRAPH_BASE}{remote_folder}/{fname}:/content"
        with open(local_path, "rb") as f:
            r = requests.put(url, headers=_graph_headers(), data=f)
        if r.status_code not in (200, 201):
            print(f"  upload failed for {fname}: {r.status_code} {r.text[:200]}")

def onedrive_download_folder(remote_folder, local_dir):
    import requests
    os.makedirs(local_dir, exist_ok=True)
    url = f"{GRAPH_BASE}{remote_folder}:/children"
    r = requests.get(url, headers=_graph_headers())
    r.raise_for_status()
    for item in r.json().get("value", []):
        if "file" in item:
            resp = requests.get(item["@microsoft.graph.downloadUrl"])
            with open(os.path.join(local_dir, item["name"]), "wb") as f:
                f.write(resp.content)

def save_to_storage(local_dir, name):
    """Persist a local folder to whichever backend is selected.
    Returns the path downstream cells should read from."""
    import shutil
    if STORAGE_BACKEND == "drive":
        dest = f"/content/drive/MyDrive/{name}"
        shutil.copytree(local_dir, dest, dirs_exist_ok=True)
        print(f"Saved to Google Drive at: {dest}")
        return dest
    else:
        remote_folder = f"/{name}"
        onedrive_upload_folder(local_dir, remote_folder)
        print(f"Uploaded to OneDrive at: {remote_folder}")
        print("Note: this session's local copy is what later cells will read from.")
        print(f"To resume in a NEW session, re-download first with: onedrive_download_folder('{remote_folder}', '{local_dir}')")
        return local_dir


## Stage 1: Image Prep
---
### 1.1 Get your source images

Reads `UPLOAD_INSTEAD` / `SOURCE_PATH` / `ONEDRIVE_SOURCE_PATH` from the config cell.

- **Upload widget** (`UPLOAD_INSTEAD = True`): a file picker appears when you run this cell.
  Select a zip, a PDF, one image, or several images — any mix is fine, everything gets sorted
  out automatically.
- **Drive / OneDrive path**: point at a `.zip`, a `.pdf`, a single image, or a folder of loose
  images already sitting in your storage. The type is auto-detected from the path.

Either way, by the end of this cell every source page is a plain image file sitting in
`input_dir`, ready for Stage 1.2.


In [ ]:
import zipfile
import shutil

!pip install -q pymupdf
import fitz  # PyMuPDF, for PDF -> image conversion

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp")

input_dir = "/content/input_images"
os.makedirs(input_dir, exist_ok=True)
output_dir = "/content/output_images"
os.makedirs(output_dir, exist_ok=True)


def is_image(fname):
    return fname.lower().endswith(IMAGE_EXTS)

def ingest_zip(local_zip_path):
    with zipfile.ZipFile(local_zip_path, "r") as zf:
        zf.extractall(input_dir)

def ingest_pdf(local_pdf_path):
    doc = fitz.open(local_pdf_path)
    base = os.path.splitext(os.path.basename(local_pdf_path))[0]
    zoom = PDF_DPI / 72
    mat = fitz.Matrix(zoom, zoom)
    for i, page in enumerate(doc):
        pix = page.get_pixmap(matrix=mat)
        pix.save(os.path.join(input_dir, f"{base}_p{i+1:03d}.png"))
    print(f"  {local_pdf_path}: {len(doc)} page(s) rasterised at {PDF_DPI} DPI")

def ingest_images_folder(local_folder):
    for fname in sorted(os.listdir(local_folder)):
        if is_image(fname):
            shutil.copy(os.path.join(local_folder, fname), os.path.join(input_dir, fname))

def ingest_any_local(local_path):
    if local_path.lower().endswith(".zip"):
        ingest_zip(local_path)
    elif local_path.lower().endswith(".pdf"):
        ingest_pdf(local_path)
    elif os.path.isdir(local_path):
        ingest_images_folder(local_path)
    elif is_image(local_path):
        shutil.copy(local_path, os.path.join(input_dir, os.path.basename(local_path)))
    else:
        raise ValueError(f"Don't know how to handle: {local_path} (expected .zip, .pdf, or an image)")


if UPLOAD_INSTEAD:
    from google.colab import files
    print("Select a zip, a PDF, or one/several images...")
    uploaded = files.upload()
    for fname in uploaded:
        ingest_any_local(fname)

elif STORAGE_BACKEND == "drive":
    ingest_any_local(SOURCE_PATH)

else:  # onedrive
    if ONEDRIVE_SOURCE_PATH.lower().endswith((".zip", ".pdf")):
        local_tmp = "/content/" + os.path.basename(ONEDRIVE_SOURCE_PATH)
        onedrive_download_file(ONEDRIVE_SOURCE_PATH, local_tmp)
        ingest_any_local(local_tmp)
    else:
        local_tmp = "/content/onedrive_source_images"
        onedrive_download_folder(ONEDRIVE_SOURCE_PATH, local_tmp)
        ingest_images_folder(local_tmp)

n_found = len([f for f in os.listdir(input_dir) if is_image(f)])
print(f"\nReady: {n_found} image(s) in {input_dir}")


### 1.2 Run prep

Fixes EXIF rotation, resizes to `target_long_edge`, applies contrast/sharpen per the settings
in the config cell, then saves each page under the 10MB cap.


In [ ]:
from PIL import Image, ImageEnhance, ImageOps, ImageFilter
from tqdm import tqdm

def resize_to_target(img, target_long_edge):
    w, h = img.size
    scale = target_long_edge / max(w, h)
    if scale >= 1:
        return img  # don't upscale a smaller image
    return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)

def enhance_image(img):
    if contrast_mode == "auto":
        return ImageOps.autocontrast(img, cutoff=autocontrast_cutoff)
    else:
        return ImageEnhance.Contrast(img).enhance(contrast_factor)

def save_under_limit(img, path, max_bytes):
    ext = os.path.splitext(path)[1].lower()

    if ext in (".jpg", ".jpeg"):
        quality = 95
        while quality >= 10:
            img.save(path, quality=quality, optimize=True)
            if os.path.getsize(path) <= max_bytes:
                return True
            quality -= 5
        return False

    else:
        current = img
        while os.path.getsize(path) > max_bytes if os.path.exists(path) else True:
            current.save(path, optimize=True)
            if os.path.getsize(path) <= max_bytes:
                return True
            w, h = current.size
            if w < 100 or h < 100:
                return False
            current = current.resize((int(w * 0.9), int(h * 0.9)), Image.LANCZOS)
        return True

for filename in tqdm(sorted(os.listdir(input_dir))):
    if is_image(filename):
        in_path = os.path.join(input_dir, filename)
        out_path = os.path.join(output_dir, filename)

        img = Image.open(in_path)
        img = ImageOps.exif_transpose(img)  # fix silent EXIF rotation before anything else

        img = resize_to_target(img, target_long_edge)

        if enhance_contrast:
            img = enhance_image(img)

        if sharpen:
            img = img.filter(ImageFilter.UnsharpMask(radius=1.5, percent=100, threshold=2))

        success = save_under_limit(img, out_path, max_size_bytes)

        status = "OK" if success else "COULD NOT HIT TARGET"
        print(f"{filename}: {img.size[0]}x{img.size[1]} -> {os.path.getsize(out_path)/1024/1024:.2f} MB [{status}]")

print("Done")


### 1.3 Save prepped images

In [ ]:
prepped_drive_folder = save_to_storage(output_dir, save_name)


## Stage 2: Transcription
---
### 2.1 Install and start Ollama


In [ ]:
!sudo apt-get install zstd
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)


### 2.2 Pull your model(s)

Pulls every model listed in `MODEL_CANDIDATES` (set in the config cell above). Each is a
multi-GB download, so double check that list before running this if you're on limited
disk/bandwidth.


In [ ]:
!pip install -q ollama
import ollama as _ollama_pkg  # import check only; the client is used properly in Stage 2.4

for m in MODEL_CANDIDATES:
    print(f"--- Pulling {m} ---")
    !ollama pull {m}


### 2.3 Point at your prepped images

Defaults to the folder Stage 1 just saved. If you're resuming a previous OneDrive session
(prepped images uploaded earlier, new runtime now), uncomment the download line to pull them
back down locally first.


In [ ]:
input_drive_folder = prepped_drive_folder

# Resuming OneDrive in a new session? Uncomment:
# input_drive_folder = "/content/prepped_images"
# onedrive_download_folder(f"/{save_name}", input_drive_folder)

output_drive_folder = input_drive_folder + "_transcriptions"
os.makedirs(output_drive_folder, exist_ok=True)

print(f"Reading images from: {input_drive_folder}")
print(f"Saving transcriptions to: {output_drive_folder}")


### 2.4 Run the batch transcription

Skips any page that already has a `.txt` file, so it's safe to re-run and it will just
continue. Each page first gets a normal whole-page attempt (with retries escalating to
`RETRY_TEMPERATURE` and falling back across `MODEL_CANDIDATES`, per the config cell). If the
result still looks weak or ink-coverage suggests the page is denser than the result reflects,
and `ENABLE_AUTO_TILING` is on, the page is automatically split into overlapping bands and
retried tiled — the better of the two results is kept.

Every model call also carries the repetition-loop guards from the config cell: a repeat
penalty at the sampling level, a lower prediction budget on sparse pages, and post-hoc
detection/truncation that trims a runaway loop (e.g. `"[illegible] [illegible] [illegible] ..."`
repeated hundreds of times) back to whatever genuine content came before it.

Writes/appends to `transcription_log.csv` in the output folder: page, character count, method
(`full-page` or `tiled`), model(s) used, ink coverage %, and whether a repetition loop was
detected and trimmed — useful for comparing models or spotting patterns across a batch without
opening every file.


In [ ]:
import glob
import base64
import csv
import re
import ollama
import numpy as np

ILLEGIBLE_PATTERN_STAGE2 = re.compile(r"\[illegible[^\]]*\]", re.IGNORECASE)

def get_image_files(folder):
    exts = ("*.jpg", "*.jpeg", "*.png", "*.webp")
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(folder, ext)))
        files.extend(glob.glob(os.path.join(folder, ext.upper())))
    return sorted(set(files))

def compute_ink_coverage_pct(image_path, dark_threshold=INK_DARK_PIXEL_THRESHOLD):
    """% of pixels darker than dark_threshold -- a cheap, content-agnostic proxy for "how much
    writing is actually on this page", independent of what the model transcribed. Used both to
    catch dense pages whose result looks thinner than the page reflects, and (in the opposite
    direction) to cap the prediction budget on sparse pages where runaway repetition has been
    observed."""
    img = Image.open(image_path).convert("L")
    arr = np.array(img)
    return 100.0 * (arr < dark_threshold).sum() / arr.size

def truncate_at_repetition(text, min_repeats=LOOP_MIN_REPEATS, max_chunk_words=LOOP_CHUNK_MAX_WORDS):
    """Detects a short word-chunk (1 to max_chunk_words words) repeating min_repeats+ times
    consecutively -- e.g. "di. di. di. di. ..." or "[illegible] [illegible] [illegible] ..." --
    and truncates the text to whatever came before the loop started. Returns
    (possibly_truncated_text, loop_detected). Genuine content before a loop is kept rather than
    the whole page being discarded."""
    words = text.split()
    n = len(words)
    for chunk_len in range(1, max_chunk_words + 1):
        i = 0
        while i + chunk_len * min_repeats <= n:
            chunk = words[i:i + chunk_len]
            repeats = 1
            j = i + chunk_len
            while j + chunk_len <= n and words[j:j + chunk_len] == chunk:
                repeats += 1
                j += chunk_len
            if repeats >= min_repeats:
                truncated = " ".join(words[:i]).strip()
                return truncated, True
            i += 1
    return text, False

def call_model(image_path, model_name, temperature):
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()

    # Sparse pages are where the repetition-loop failure has actually been observed, so they
    # get a much smaller prediction budget -- a second line of defence alongside repeat_penalty
    # and post-hoc truncation below, in case a loop isn't caught by either.
    ink_pct = compute_ink_coverage_pct(image_path)
    predict_budget = SPARSE_NUM_PREDICT_CAP if ink_pct <= SPARSE_INK_THRESHOLD_PCT else NUM_PREDICT

    response = ollama.generate(
        model=model_name,
        prompt=PROMPT,
        images=[img_b64],
        options={
            "temperature": temperature,
            "num_ctx": NUM_CTX,
            "num_predict": predict_budget,
            "repeat_penalty": REPEAT_PENALTY,
            "repeat_last_n": REPEAT_LAST_N,
        },
    )
    text = response["response"].strip()
    text, loop_detected = truncate_at_repetition(text)
    return text, loop_detected

def transcribe_with_retries(image_path, model_name, max_attempts=MAX_ATTEMPTS):
    last_text, last_loop = "", False
    for attempt in range(1, max_attempts + 1):
        # v5: only the FIRST attempt uses TEMPERATURE (0.0 by default). Every attempt after
        # that uses RETRY_TEMPERATURE, so a retry can actually differ from the previous attempt
        # instead of deterministically reproducing the same failure.
        temperature = TEMPERATURE if attempt == 1 else RETRY_TEMPERATURE
        try:
            text, loop_detected = call_model(image_path, model_name, temperature)
        except Exception as e:
            print(f"    [{model_name}] attempt {attempt} (temp={temperature}): error ({e})")
            continue

        if loop_detected:
            print(f"    [{model_name}] attempt {attempt}: repetition loop detected and trimmed, {len(text)} char(s) of real content kept")

        if len(text) >= MIN_CHARS:
            return text, loop_detected
        last_text, last_loop = text, loop_detected
        print(f"    [{model_name}] attempt {attempt} (temp={temperature}): only {len(text)} chars, retrying...")
    return last_text, last_loop

def transcribe_with_fallback(image_path):
    text, loop_detected = "", False
    used_model = MODEL_CANDIDATES[0]
    for model_name in MODEL_CANDIDATES:
        text, loop_detected = transcribe_with_retries(image_path, model_name)
        used_model = model_name
        if len(text) >= MIN_CHARS:
            break
        print(f"    [{model_name}] exhausted retries, trying next model if available")
    return text, used_model, loop_detected

def split_into_tiles(image_path, n_tiles=TILE_COUNT, overlap_frac=TILE_OVERLAP_FRAC):
    """Split a page into n_tiles overlapping horizontal bands. Used for pages that still look
    weak after a whole-page attempt -- each band has far fewer lines to resolve than the full
    page at the same pixel density, which is the actual bottleneck on dense pages."""
    img = Image.open(image_path)
    w, h = img.size
    tile_h = h / n_tiles
    overlap_px = int(tile_h * overlap_frac)

    tile_dir = os.path.join(output_drive_folder, "_tiles_tmp")
    os.makedirs(tile_dir, exist_ok=True)
    stem = os.path.splitext(os.path.basename(image_path))[0]

    tile_paths = []
    for i in range(n_tiles):
        top = max(0, int(i * tile_h) - overlap_px)
        bottom = min(h, int((i + 1) * tile_h) + overlap_px)
        tile = img.crop((0, top, w, bottom))
        tile_path = os.path.join(tile_dir, f"{stem}_tile{i+1}.jpg")
        tile.save(tile_path, quality=95)
        tile_paths.append(tile_path)
    return tile_paths

def transcribe_tiled(image_path):
    """Transcribe each tile independently (full retry/fallback logic per tile, including the
    repetition guard) and stitch the results together. Note: overlap between tiles can
    duplicate a line near each seam -- worth a manual spot-check on tiled pages rather than
    assuming a perfectly clean stitch."""
    tile_paths = split_into_tiles(image_path)
    tile_texts, tile_models, tile_loops = [], [], []
    for tile_path in tile_paths:
        t_text, t_model, t_loop = transcribe_with_fallback(tile_path)
        tile_texts.append(t_text)
        tile_models.append(t_model)
        tile_loops.append(t_loop)
    combined = "\n\n".join(t for t in tile_texts if t.strip())
    return combined, tile_models, any(tile_loops)

def needs_tiling(text, ink_pct):
    """True if the result looks weak outright, OR if the page visually looks denser than the
    result reflects (plenty of ink, but thin content once [illegible] placeholders are
    stripped)."""
    if len(text) < MIN_CHARS:
        return True
    real_content = ILLEGIBLE_PATTERN_STAGE2.sub("", text).strip()
    if ink_pct >= DENSE_INK_THRESHOLD_PCT and len(real_content) < DENSE_PAGE_MIN_CHARS:
        return True
    return False

image_files = get_image_files(input_drive_folder)

if not image_files:
    print(f"No image files found in {input_drive_folder}. Check the path above.")
else:
    pending = [
        f for f in image_files
        if not os.path.exists(os.path.join(output_drive_folder, os.path.splitext(os.path.basename(f))[0] + ".txt"))
    ]

    if not pending:
        print(f"All {len(image_files)} pages already transcribed in {output_drive_folder}.")
        print("Delete a .txt file there if you want to re-transcribe that page.")
    else:
        batch = pending[:BATCH_SIZE]
        print(f"Found {len(image_files)} total pages, {len(pending)} pending.")
        print(f"Processing {len(batch)} pages this run (BATCH_SIZE={BATCH_SIZE}).\n")

        log_rows = []

        for filename in tqdm(batch):
            page_name = os.path.basename(filename)
            txt_name = os.path.splitext(page_name)[0] + ".txt"
            txt_path = os.path.join(output_drive_folder, txt_name)

            ink_pct = compute_ink_coverage_pct(filename)
            text, used_model, loop_detected = transcribe_with_fallback(filename)
            method = "full-page"
            models_used = used_model

            if ENABLE_AUTO_TILING and needs_tiling(text, ink_pct):
                print(f"    ink coverage {ink_pct:.2f}% looks denser than this result reflects -- retrying tiled...")
                tiled_text, tile_models, tiled_loop = transcribe_tiled(filename)
                if len(tiled_text) > len(text):
                    text = tiled_text
                    method = "tiled"
                    models_used = "+".join(tile_models)
                    loop_detected = tiled_loop

            with open(txt_path, "w", encoding="utf-8") as f:
                f.write(text)

            flag = "" if len(text) >= MIN_CHARS else "  [STILL EMPTY/SHORT — needs a look]"
            loop_note = "  [repetition loop trimmed]" if loop_detected else ""
            print(f"{page_name}: {len(text)} chars via {models_used} [{method}], ink={ink_pct:.2f}%{flag}{loop_note}")

            log_rows.append({
                "page": page_name, "chars": len(text), "method": method,
                "model": models_used, "ink_pct": round(ink_pct, 2),
                "loop_detected": loop_detected,
            })

        log_path = os.path.join(output_drive_folder, "transcription_log.csv")
        write_header = not os.path.exists(log_path)
        with open(log_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["page", "chars", "method", "model", "ink_pct", "loop_detected"])
            if write_header:
                writer.writeheader()
            for row in log_rows:
                writer.writerow(row)

        remaining = len(pending) - len(batch)
        print("\nDone this run.")
        if remaining > 0:
            print(f"{remaining} pages remaining — just re-run this cell to continue.")
        else:
            print("All pages transcribed.")


### 2.5 Flag any remaining weak results

Scans every saved `.txt` for two problems, plus one more from `transcription_log.csv`:
- anything under `MIN_CHARS` (as before) -- outright empty/failed pages
- **dense mismatches**: pages whose image has ink coverage above `DENSE_INK_THRESHOLD_PCT` but
  whose real content -- after stripping `[illegible]`/`[illegible line]` placeholders -- is
  under `DENSE_PAGE_MIN_CHARS`. These clear the plain character-count check (a page full of
  "[illegible]" placeholders can easily total 20+ characters) while containing almost nothing
  useful.
- **trimmed repetition loops**: pages where `loop_detected` is `True` in the log. The saved
  `.txt` already has the loop cut off, so this doesn't necessarily mean the page needs
  re-running — but it's worth a quick look to confirm the kept content isn't itself cut short
  mid-thought, especially on pages where very little real content preceded the loop.


In [ ]:
import re

ILLEGIBLE_PATTERN_QA = re.compile(r"\[illegible[^\]]*\]", re.IGNORECASE)

def find_image_for_stem(folder, stem):
    for ext in (".jpg", ".jpeg", ".png", ".webp", ".JPG", ".JPEG", ".PNG", ".WEBP"):
        candidate = os.path.join(folder, stem + ext)
        if os.path.exists(candidate):
            return candidate
    return None

weak = []
dense_mismatch = []

for txt_path in sorted(glob.glob(os.path.join(output_drive_folder, "*.txt"))):
    if os.path.basename(txt_path) == "all_pages_transcription.txt":
        continue
    with open(txt_path, "r", encoding="utf-8") as f:
        content = f.read()

    if len(content) < MIN_CHARS:
        weak.append(os.path.basename(txt_path))
        continue

    stem = os.path.splitext(os.path.basename(txt_path))[0]
    image_path = find_image_for_stem(input_drive_folder, stem)
    if image_path is None:
        continue  # can't check ink coverage without the source image -- skip this check for it

    ink_pct = compute_ink_coverage_pct(image_path)
    real_content = ILLEGIBLE_PATTERN_QA.sub("", content).strip()
    if ink_pct >= DENSE_INK_THRESHOLD_PCT and len(real_content) < DENSE_PAGE_MIN_CHARS:
        dense_mismatch.append((os.path.basename(txt_path), round(ink_pct, 2), len(real_content)))

if weak:
    print(f"{len(weak)} page(s) still weak/empty:")
    for w in weak:
        print(f"  - {w}")
    print()

if dense_mismatch:
    print(f"{len(dense_mismatch)} page(s) look denser than their transcription reflects:")
    for name, ink_pct, real_chars in dense_mismatch:
        print(f"  - {name}: ink coverage {ink_pct}%, only {real_chars} real char(s) after stripping [illegible] tags")
    print()

# Repetition loops -- read from the log rather than re-detecting from the (already-trimmed)
# saved .txt, since the log is the only place that still records it happened.
log_path = os.path.join(output_drive_folder, "transcription_log.csv")
looped = []
if os.path.exists(log_path):
    with open(log_path, "r", newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row.get("loop_detected", "").strip() in ("True", "true", "1"):
                looped.append((row["page"], row["chars"]))

if looped:
    print(f"{len(looped)} page(s) had a repetition loop detected and trimmed:")
    for name, chars in looped:
        print(f"  - {name}: {chars} char(s) kept after trimming")
    print("  (worth a quick check that the kept content isn't cut short mid-thought)")
    print()

if not (weak or dense_mismatch or looped):
    print("No weak, mismatched, or looped transcriptions found.")
elif weak or dense_mismatch:
    print("Delete the weak/mismatched .txt files above and re-run the transcription cell to retry them")
    print("(tiling will kick in automatically for dense_mismatch pages if ENABLE_AUTO_TILING is True).")


### 2.6 Combine into a single file

In [ ]:
combined_path = os.path.join(output_drive_folder, "all_pages_transcription.txt")

all_txts = sorted(
    f for f in glob.glob(os.path.join(output_drive_folder, "*.txt"))
    if os.path.basename(f) != "all_pages_transcription.txt"
)

with open(combined_path, "w", encoding="utf-8") as out:
    for t in all_txts:
        page_label = os.path.splitext(os.path.basename(t))[0]
        with open(t, "r", encoding="utf-8") as f:
            content = f.read()
        out.write(f"--- {page_label} ---\n{content}\n\n")

print(f"Combined {len(all_txts)} pages into: {combined_path}")


### 2.7 Persist transcriptions

For Drive, your `.txt` files are already sitting in Drive — nothing further needed. For
OneDrive, this uploads the transcriptions folder (they've only existed locally in this session
until now).


In [ ]:
if STORAGE_BACKEND == "onedrive":
    transcription_save_name = os.path.basename(output_drive_folder)
    save_to_storage(output_drive_folder, transcription_save_name)
else:
    print(f"Already saved in Google Drive at: {output_drive_folder}")


## Stage 3: Cataloguing (optional)
---
Drafts an item-level **ISAD(G)** catalogue record for each page, generated from its
transcription: title, dates, extent, creator, scope and content, language/scripts, physical
characteristics, access points (persons/places/subjects/organisations), a confidence estimate
per field, and a list of points flagged for review (including anything potentially sensitive).

This uses a separate **text-only** model reading the transcribed text — not the original
image. That means anything that genuinely needs a look at the image (physical characteristics,
handwriting/script style, true dimensions) should come back blank with low confidence and a
verification flag, rather than a guess. **Every record here is a draft** — a human cataloguer
should review each one, especially anything flagged, before it's used.

Controlled by `GENERATE_SUMMARIES` in the config cell — leave it `False` to skip this stage
entirely.


### 3.1 Pull the summary model

In [ ]:
if GENERATE_SUMMARIES:
    print(f"--- Pulling {SUMMARY_MODEL} ---")
    !ollama pull {SUMMARY_MODEL}
else:
    print("GENERATE_SUMMARIES is False in the config cell — skipping the cataloguing stage.")


### 3.2 Generate catalogue records

One JSON record per page, saved alongside a mirror of the transcriptions folder. Like Stage 2,
this skips any page that already has a record, so it's safe to re-run — just delete a page's
`.json` file first if you want it regenerated.

Pages whose transcription is empty or just "[illegible]" placeholders are skipped rather than
catalogued (see `CATALOGUE_MIN_CONTENT_CHARS` in the config cell) — logged to
`skipped_illegible.txt` in the catalogue folder rather than silently dropped.


In [ ]:
import json
import re
import csv

catalogue_dir = input_drive_folder + "_catalogue"

ILLEGIBLE_PATTERN = re.compile(r"\[illegible[^\]]*\]", re.IGNORECASE)

def has_catalogueable_content(transcription_text):
    """False for empty pages, or pages that are just [illegible]/[illegible line] placeholders
    with nothing else of substance."""
    stripped = transcription_text.strip()
    if not stripped:
        return False
    remaining = ILLEGIBLE_PATTERN.sub("", stripped).strip()
    return len(remaining) >= CATALOGUE_MIN_CONTENT_CHARS

def extract_json_record(raw_text):
    """Pull a JSON object out of a model response, tolerating ```json fences or stray text
    around it."""
    text = raw_text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\n?", "", text)
        text = re.sub(r"\n?```$", "", text)
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        text = match.group(0)
    return json.loads(text)

def catalogue_page(page_name, transcription_text):
    full_prompt = SUMMARY_INSTRUCTIONS + f"\n\n---\nPAGE FILENAME: {page_name}\nTRANSCRIBED TEXT:\n{transcription_text}\n---\n"
    last_error = None
    for attempt in range(1, SUMMARY_MAX_ATTEMPTS + 1):
        try:
            response = ollama.generate(
                model=SUMMARY_MODEL,
                prompt=full_prompt,
                options={
                    "temperature": SUMMARY_TEMPERATURE,
                    "num_ctx": SUMMARY_NUM_CTX,
                    "num_predict": SUMMARY_NUM_PREDICT,
                },
            )
            record = extract_json_record(response["response"])
            record["page"] = page_name
            return record, None
        except Exception as e:
            last_error = str(e)
            print(f"    [{SUMMARY_MODEL}] attempt {attempt}: could not parse a valid record ({last_error})")
    return None, last_error

if GENERATE_SUMMARIES:
    os.makedirs(catalogue_dir, exist_ok=True)

    txt_files = sorted(
        f for f in glob.glob(os.path.join(output_drive_folder, "*.txt"))
        if os.path.basename(f) != "all_pages_transcription.txt"
    )

    not_yet_done = [
        t for t in txt_files
        if not os.path.exists(os.path.join(catalogue_dir, os.path.splitext(os.path.basename(t))[0] + ".json"))
    ]

    pending = []
    skipped_illegible = []
    for txt_path in not_yet_done:
        with open(txt_path, "r", encoding="utf-8") as f:
            transcription_text = f.read()
        if has_catalogueable_content(transcription_text):
            pending.append(txt_path)
        else:
            skipped_illegible.append(os.path.splitext(os.path.basename(txt_path))[0])

    if skipped_illegible:
        skipped_path = os.path.join(catalogue_dir, "skipped_illegible.txt")
        with open(skipped_path, "w", encoding="utf-8") as f:
            f.write(f"Pages skipped as empty/illegible (< {CATALOGUE_MIN_CONTENT_CHARS} chars of real content):\n")
            f.write("\n".join(skipped_illegible))
        print(f"Skipped {len(skipped_illegible)} empty/illegible page(s) — not catalogued. See {skipped_path}\n")

    if not txt_files:
        print(f"No transcriptions found in {output_drive_folder} — run Stage 2 first.")
    elif not pending:
        print(f"No pages pending catalogue records ({len(txt_files)} total transcribed, {len(skipped_illegible)} skipped as illegible/empty).")
        print("Delete a .json file in the catalogue folder if you want to regenerate that page's record.")
    else:
        print(f"Found {len(txt_files)} transcribed pages, {len(pending)} pending catalogue records.\n")
        failed = []
        for txt_path in tqdm(pending):
            page_name = os.path.splitext(os.path.basename(txt_path))[0]
            with open(txt_path, "r", encoding="utf-8") as f:
                transcription_text = f.read()

            record, error = catalogue_page(page_name, transcription_text)
            json_path = os.path.join(catalogue_dir, page_name + ".json")

            if record is not None:
                with open(json_path, "w", encoding="utf-8") as f:
                    json.dump(record, f, indent=2, ensure_ascii=False)
                n_flags = len(record.get("flags", []))
                flag_note = f", {n_flags} flag(s)" if n_flags else ""
                print(f"{page_name}: catalogued{flag_note}")
            else:
                failed.append(page_name)
                print(f"{page_name}: FAILED to produce a valid record ({error})")

        print("\nDone this run.")
        if failed:
            print(f"{len(failed)} page(s) failed to produce a valid record: {', '.join(failed)}")
            print("Re-run this cell to retry them.")
else:
    print("GENERATE_SUMMARIES is False — skipping.")


### 3.3 Flag records for review

Pulls every flag out of every record into one place — split into general verification points
and anything marked potentially sensitive — so you don't have to open each JSON file to find
them.


In [ ]:
if GENERATE_SUMMARIES:
    catalogue_files = sorted(glob.glob(os.path.join(catalogue_dir, "*.json")))
    review_lines = []
    sensitivity_count = 0
    verification_count = 0

    for jf in catalogue_files:
        with open(jf, "r", encoding="utf-8") as f:
            record = json.load(f)
        flags = record.get("flags", [])
        if not flags:
            continue
        page = record.get("page", os.path.splitext(os.path.basename(jf))[0])
        review_lines.append(f"--- {page} ---")
        for flag in flags:
            ftype = flag.get("type", "verification")
            note = flag.get("note", "")
            if ftype == "sensitivity":
                review_lines.append(f"  [SENSITIVITY] {note}")
                sensitivity_count += 1
            else:
                review_lines.append(f"  [verification] {note}")
                verification_count += 1

    review_path = os.path.join(catalogue_dir, "flagged_for_review.txt")
    if review_lines:
        with open(review_path, "w", encoding="utf-8") as f:
            f.write("\n".join(review_lines))
        print(f"{verification_count} verification point(s) and {sensitivity_count} sensitivity flag(s) across {len(catalogue_files)} record(s).")
        print(f"Full list written to: {review_path}")
        if sensitivity_count:
            print("\nReview sensitivity-flagged records before sharing or granting wider access.")
    else:
        print(f"No records flagged for review, out of {len(catalogue_files)} record(s).")
else:
    print("GENERATE_SUMMARIES is False — skipping.")


### 3.4 Combine into a catalogue

Bundles every page's record into one JSON array (for re-import/scripting) and one flattened
CSV (for a quick spreadsheet review, or import into archival management software).


In [ ]:
if GENERATE_SUMMARIES:
    catalogue_files = sorted(glob.glob(os.path.join(catalogue_dir, "*.json")))
    combined_records = [json.load(open(jf, "r", encoding="utf-8")) for jf in catalogue_files]

    combined_json_path = os.path.join(catalogue_dir, "all_catalogue_records.json")
    with open(combined_json_path, "w", encoding="utf-8") as f:
        json.dump(combined_records, f, indent=2, ensure_ascii=False)

    csv_fields = [
        "page", "title", "dates_as_written", "dates_normalised", "extent", "creator",
        "scope_and_content", "language_scripts", "physical_characteristics",
        "access_points_persons", "access_points_places", "access_points_subjects",
        "access_points_organisations", "confidence_title", "confidence_dates",
        "confidence_extent", "confidence_creator", "confidence_scope_and_content",
        "confidence_language_scripts", "confidence_physical_characteristics",
        "confidence_access_points", "flags_summary",
    ]
    combined_csv_path = os.path.join(catalogue_dir, "all_catalogue_records.csv")
    with open(combined_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=csv_fields)
        writer.writeheader()
        for r in combined_records:
            dates = r.get("dates", {}) or {}
            ap = r.get("access_points", {}) or {}
            conf = r.get("confidence", {}) or {}
            flags = r.get("flags", []) or []
            flags_summary = "; ".join(f"[{fl.get('type', 'verification')}] {fl.get('note', '')}" for fl in flags)
            writer.writerow({
                "page": r.get("page", ""),
                "title": r.get("title", ""),
                "dates_as_written": dates.get("as_written", ""),
                "dates_normalised": dates.get("normalised", ""),
                "extent": r.get("extent", ""),
                "creator": r.get("creator", ""),
                "scope_and_content": r.get("scope_and_content", ""),
                "language_scripts": r.get("language_scripts", ""),
                "physical_characteristics": r.get("physical_characteristics", ""),
                "access_points_persons": "; ".join(ap.get("persons", [])),
                "access_points_places": "; ".join(ap.get("places", [])),
                "access_points_subjects": "; ".join(ap.get("subjects", [])),
                "access_points_organisations": "; ".join(ap.get("organisations", [])),
                "confidence_title": conf.get("title", ""),
                "confidence_dates": conf.get("dates", ""),
                "confidence_extent": conf.get("extent", ""),
                "confidence_creator": conf.get("creator", ""),
                "confidence_scope_and_content": conf.get("scope_and_content", ""),
                "confidence_language_scripts": conf.get("language_scripts", ""),
                "confidence_physical_characteristics": conf.get("physical_characteristics", ""),
                "confidence_access_points": conf.get("access_points", ""),
                "flags_summary": flags_summary,
            })

    print(f"Combined {len(combined_records)} record(s) into:")
    print(f"  {combined_json_path}")
    print(f"  {combined_csv_path}")
else:
    print("GENERATE_SUMMARIES is False — skipping.")


### 3.5 Persist catalogue records

For Drive, these are already sitting in Drive — nothing further needed. For OneDrive, this
uploads the catalogue folder.


In [ ]:
if GENERATE_SUMMARIES:
    if STORAGE_BACKEND == "onedrive":
        catalogue_save_name = os.path.basename(catalogue_dir)
        save_to_storage(catalogue_dir, catalogue_save_name)
    else:
        print(f"Already saved in Google Drive at: {catalogue_dir}")
else:
    print("GENERATE_SUMMARIES is False — skipping.")
